# Model Development — Robust Multi-Objective Distribution Network Design

**Case study:** Personal care manufacturer distribution network, Java, Indonesia.
**Scope:** Geography = Java only. Products = Beauty + Personal Care segments only.
**Data:** cleaned in `Validation.ipynb`, files under `D:\KLTN\Data Dictionary\Model - Data used\`.

This notebook is structured as a report chapter: each section has a narrative part (problem statement, notation, math model, method) for you to write, followed by the code that implements it.


---
# 1. Problem Statement

> ✍️ **Write here:** the research problem in plain language — what network decisions are being made (which facilities stay open, how goods flow), what makes it multi-objective (cost vs. coverage vs. CO2), why demand uncertainty matters, and why a robust optimization + AUGMECON2 + TOPSIS approach was chosen. This is the narrative version of what you already drafted for your advisor.

*(placeholder — fill in)*


---
# 2. Notation

## 2.1 Sets and Indices

| Symbol | Description | Source |
|---|---|---|
| $j \in J$ | Facilities (FIXED $\cup$ DECISION) | `FacilityMaster.parquet` |
| $k \in K$ | Demand clusters (ShipToGroupID) | `shipto_crosswalk.parquet` |
| $t \in T$ | Time period (month), if time-indexed | `b2b_so_final.parquet` |

> ✍️ **Write here:** add/adjust any other sets you introduce (e.g. product groups, scenarios).

## 2.2 Parameters

| Symbol | Description | Source |
|---|---|---|
| $\bar d_k$ | Nominal (mean) monthly demand at cluster $k$, in pallets | Section 4.2 |
| $\hat d_k$ | Demand deviation at cluster $k$ (uncertainty budget) | Section 4.2 |
| $Dist_{jk}$ | Distance, facility $j$ to cluster $k$ | Section 4.4 |
| $D_{max}$ | Max SLA distance (150 km, Java B2B) | Paragon Assumption Doc §4.7 |
| $CostLastMile_{jk}$ | Last-mile delivery cost, IDR/pallet | `ENO_CostLastMileB2B.parquet` |
| $CostTransfer_{jj'}$ | Facility-to-facility transfer cost, IDR/pallet | `ENO_CostTransfer.parquet` |
| $CostSupply_{fj}$ | Factory-to-facility supply cost, IDR/pallet | `ENO_CostSupply.parquet` |
| $DOS_{j,pg}$ | Days of stock, facility × product group | `ENO_DOS.parquet` |
| $WACC$ | Cost of capital for IOC (10%) | Model.md, confirmed by Paragon |
| $CO2_{jk}$ | Emissions factor, facility → cluster | `Data Dictionary/CO2/` |

> ✍️ **Write here:** any additional parameters (capacity, fixed cost, etc.) as you finalize the formulation.

## 2.3 Decision Variables

| Symbol | Type | Description |
|---|---|---|
| $Y_j$ | Binary | 1 if facility $j$ (DECISION set) is open |
| $X_{jk}$ | Continuous $\ge 0$ | Flow (pallets) from facility $j$ to cluster $k$ |
| $U_k$ | Binary/continuous | Unmet/uncovered demand indicator at cluster $k$ |

> ✍️ **Write here:** finalize exact variable set once the constraint structure is settled.


---
# 3. Mathematical Model

## 3.1 Objective Functions

**$f_1$ — Total Cost (minimize)**

$$
f_1 = \sum_{j,k} CostLastMile_{jk} \cdot X_{jk} + \sum_{j,j'} CostTransfer_{jj'} \cdot X_{jj'} + \sum_{f,j} CostSupply_{fj} \cdot X_{fj} + \sum_j FixedStorageCost_j \cdot Y_j + IOC
$$

> ✍️ **Write here:** finalize the exact cost terms (which are linear in $X$ vs. fixed per $Y_j$), and the IOC formula (uses $DOS$ and $WACC$).

**$f_2$ — Uncovered Demand (minimize)**

$$
f_2 = \sum_k \bar d_k \cdot U_k
$$

**$f_3$ — CO2 Emissions (minimize)**

$$
f_3 = \sum_{j,k} CO2_{jk} \cdot X_{jk}
$$

> ✍️ **Write here:** confirm emissions factor source/units (Global Logistics Emissions Council Framework).

## 3.2 Constraints

> ✍️ **Write here** each constraint in LaTeX as you finalize it. Skeleton below — fill in:

**Flow balance**
$$
\sum_j X_{jk} = \bar d_k \quad \forall k
$$

**Coverage**
$$
U_k \le 1 - \sum_{j:\, Dist_{jk} \le D_{max}} Y_j \quad \forall k
$$

**Robust demand constraint (Bertsimas & Sim, 2004)**
$$
\text{(write the budget-of-uncertainty reformulation here, using } \hat d_k \text{ and } \Gamma \text{)}
$$

**Facility status**
$$
Y_j = 1 \quad \forall j \in \text{FIXED (NDC)}
$$

**Domain**
$$
Y_j \in \{0,1\}, \quad X_{jk} \ge 0
$$


---
# 4. Data Preparation

## 4.1 Load Cleaned Data

| File | Contents |
|---|---|
| `customer_master_clean.parquet` | Customer ship-to locations (Java only) |
| `b2b_so_final.parquet` | B2B sales orders Jan–Nov 2025 (Java + Beauty/Personal Care) |
| `shipto_crosswalk.parquet` | `ShipToID` → `ShipToGroupID` mapping |
| `product_master_clean.parquet` | Product master (Beauty+Personal Care, `KGPerPallet` fixed) |
| `FacilityMaster.parquet` | 27 facilities in Java |


In [1]:
# ===== Import libraries =====
import pandas as pd
import numpy as np
import os

# ===== Paths =====
DATA_PATH = r"D:\KLTN\Data Dictionary\Model - Data used"

# Root chứa raw scenario data của CEL (Baseline Constrained + Unconstrained/Greenfield)
# Dùng riêng cho các bảng cost ở Section 4.5/4.6 - xem note chọn scenario ở đó
RAW_DATA_PATH = r"D:\KLTN\Data Dictionary\Modeling\Data"

# ===== Load các bảng nhỏ (customer/shipto/product/facility) =====
customer_master = pd.read_parquet(os.path.join(DATA_PATH, "customer_master_clean.parquet"))
shipto_crosswalk = pd.read_parquet(os.path.join(DATA_PATH, "shipto_crosswalk.parquet"))
product_master = pd.read_parquet(os.path.join(DATA_PATH, "product_master_clean.parquet"))
facility_master = pd.read_parquet(os.path.join(DATA_PATH, "FacilityMaster.parquet"))

# ===== Load b2b_so (lấy hết 27 cột cho chắc, ko lọc cột nữa) =====
b2b_so = pd.read_parquet(os.path.join(DATA_PATH, "b2b_so_final.parquet"))

# ===== Giảm RAM cho b2b_so bằng cách convert cột lặp lại nhiều sang dtype "category" =====
# Ko bớt cột, nhưng vẫn tối ưu được RAM cho các cột có giá trị lặp lại rất nhiều
# (ShipToID/ProductID chỉ có vài nghìn giá trị unique trên 14.5 triệu dòng,
#  OrderStatus chỉ có vài giá trị như Done/Canceled) -> convert category ko mất thông tin gì
b2b_so["ShipToID"] = b2b_so["ShipToID"].astype("category")
b2b_so["ProductID"] = b2b_so["ProductID"].astype("category")
b2b_so["OrderStatus"] = b2b_so["OrderStatus"].astype("category")

## 4.2 Demand Aggregation ($\bar d_k$, $\hat d_k$)

1. Merge `b2b_so` with `shipto_crosswalk` on `ShipToID` → attach `ShipToGroupID`.
2. Merge with `product_master` on `ProductID` → get `KGPerPallet` → compute `PalletQty`.
3. Aggregate `PalletQty` by `ShipToGroupID` × month (11 months).
4. Compute $\bar d_k$ (mean) and $\hat d_k$ (deviation) per cluster.


In [2]:
# TODO: merge b2b_so -> shipto_crosswalk -> product_master, compute PalletQty
# TODO: aggregate by ShipToGroupID x month
# TODO: compute d_bar (mean) and d_hat (deviation) per ShipToGroupID

# ===== BƯỚC 1: Map ShipToID -> ShipToGroupID (demand node của model) =====
b2b_agg = b2b_so.merge(shipto_crosswalk, on="ShipToID", how="left")

# ===== BƯỚC 2: Convert kg -> pallet bằng KGPerPallet của từng product =====
# Demand lấy hết mọi order, kể cả Canceled - ko lọc OrderStatus
b2b_agg = b2b_agg.merge(product_master[["ProductID", "KGPerPallet"]], on="ProductID", how="left")
b2b_agg["QtyInPallet"] = b2b_agg["QtyOrderedInKG"] / b2b_agg["KGPerPallet"]

# ===== BƯỚC 3: Lấy tháng từ OrderedTimestamp để tính demand theo tháng =====
b2b_agg["OrderMonth"] = b2b_agg["OrderedTimestamp"].dt.to_period("M")

# ===== BƯỚC 4: Aggregate demand (pallet) theo (ShipToGroupID, tháng) =====
monthly_demand = b2b_agg.groupby(["ShipToGroupID", "OrderMonth"])["QtyInPallet"].sum().reset_index()

# ===== BƯỚC 5: Điền đủ 0 cho node/tháng ko có order (0 là data point thật, ko phải thiếu) =====
all_nodes = shipto_crosswalk["ShipToGroupID"].unique()
all_months = monthly_demand["OrderMonth"].unique()
full_index = pd.MultiIndex.from_product([all_nodes, all_months], names=["ShipToGroupID", "OrderMonth"])
monthly_demand_full = (
    monthly_demand.set_index(["ShipToGroupID", "OrderMonth"])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

# ===== BƯỚC 6: Tính d_bar (nominal) và d_hat (deviation) cho từng node =====
demand_agg = monthly_demand_full.groupby("ShipToGroupID")["QtyInPallet"].agg(
    d_bar="mean",
    d_max="max"
).reset_index()
demand_agg["d_hat"] = (demand_agg["d_max"] - demand_agg["d_bar"]).clip(lower=0)

# ===== SANITY CHECK =====
print("Shape demand_agg:", demand_agg.shape)
print(demand_agg.head(10))
print("\nMô tả thống kê d_bar / d_hat:")
print(demand_agg[["d_bar", "d_hat"]].describe())

C:\Users\USER\AppData\Local\Temp\ipykernel_13932\3855468875.py:14: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  b2b_agg["OrderMonth"] = b2b_agg["OrderedTimestamp"].dt.to_period("M")


Shape demand_agg: (1763, 4)
                         ShipToGroupID       d_bar        d_max       d_hat
0                           105695-002  134.691707   192.858195   58.166488
1                           118438-598  474.988263   916.489769  441.501506
2                           124624-000   95.059856   171.497252   76.437396
3                           128112-002   44.163454    61.779295   17.615841
4                           138789-000  100.016596   212.309713  112.293118
5                           150794-001   68.519253   113.816983   45.297730
6                           230001-001  826.089194  1409.251768  583.162574
7    ID_JV_Banten_Kota Cilegon_Cibeber    0.668009     4.123876    3.455867
8    ID_JV_Banten_Kota Cilegon_Cilegon    0.231137     0.342797    0.111659
9  ID_JV_Banten_Kota Cilegon_Citangkil    0.053710     0.236421    0.182711

Mô tả thống kê d_bar / d_hat:
             d_bar        d_hat
count  1763.000000  1763.000000
mean      3.899830     2.830026
std      

## 4.3 Cluster Compactness Validation

Haversine distance between member `ShipToID`s and their cluster centroid, to support the methodology write-up (District-level grouping doesn't distort geography much).


In [9]:
# TODO: Haversine distance function
# TODO: compute per-cluster weighted_avg_dist_km
import numpy as np

# ===== BƯỚC 1: Hàm tính khoảng cách Haversine (km) giữa 2 điểm lat/long =====
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # bán kính Trái Đất (km)
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

# ===== BƯỚC 2: Ghép tọa độ (CentroidLat/CentroidLong) của từng ShipToID vào shipto_crosswalk =====
cluster_coords = shipto_crosswalk.merge(
    customer_master[["LocationID", "CentroidLat", "CentroidLong"]],
    left_on="ShipToID", right_on="LocationID", how="left"
)

# ===== BƯỚC 3: Tính tâm cụm (centroid) = trung bình tọa độ các ShipToID trong cùng ShipToGroupID =====
cluster_centroids = cluster_coords.groupby("ShipToGroupID").agg(
    centroid_lat=("CentroidLat", "mean"),
    centroid_lon=("CentroidLong", "mean"),
    n_members=("ShipToID", "nunique")
).reset_index()

# ===== BƯỚC 4: Ghép tâm cụm ngược lại, tính khoảng cách từng ShipToID -> tâm cụm của nó =====
cluster_coords = cluster_coords.merge(cluster_centroids, on="ShipToGroupID", how="left")
cluster_coords["dist_to_centroid_km"] = haversine(
    cluster_coords["CentroidLat"], cluster_coords["CentroidLong"],
    cluster_coords["centroid_lat"], cluster_coords["centroid_lon"]
)

# ===== BƯỚC 5: Tính "bán kính cụm" (max/mean khoảng cách) cho mỗi ShipToGroupID =====
# Cụm standalone (1 member, ShipToGroupID = ShipToID) sẽ tự động ra bán kính = 0 - đúng như kỳ vọng
cluster_radius = cluster_coords.groupby("ShipToGroupID").agg(
    n_members=("ShipToID", "nunique"),
    max_radius_km=("dist_to_centroid_km", "max"),
    mean_radius_km=("dist_to_centroid_km", "mean")
).reset_index()

# ===== SANITY CHECK =====
print("Shape cluster_radius:", cluster_radius.shape)
print(cluster_radius.head(10))
print("\nMô tả thống kê bán kính cụm (km):")
print(cluster_radius[["max_radius_km", "mean_radius_km"]].describe())

# ===== Đếm số cụm có bán kính bất thường lớn (VD > 20km, coi như ngưỡng "không compact") =====
n_outlier_clusters = (cluster_radius["max_radius_km"] > 20).sum()
print(f"\nSố cụm có bán kính > 20km: {n_outlier_clusters} / {len(cluster_radius)}")

Shape cluster_radius: (1763, 4)
                         ShipToGroupID  n_members  max_radius_km  \
0                           105695-002          1       0.000000   
1                           118438-598          1       0.000000   
2                           124624-000          1       0.000000   
3                           128112-002          1       0.000000   
4                           138789-000          1       0.000000   
5                           150794-001          1       0.000000   
6                           230001-001          1       0.000000   
7    ID_JV_Banten_Kota Cilegon_Cibeber          8       0.693171   
8    ID_JV_Banten_Kota Cilegon_Cilegon          4       0.671852   
9  ID_JV_Banten_Kota Cilegon_Citangkil          8       1.608656   

   mean_radius_km  
0        0.000000  
1        0.000000  
2        0.000000  
3        0.000000  
4        0.000000  
5        0.000000  
6        0.000000  
7        0.693171  
8        0.335926  
9        0.889924  

## 4.4 Facility Classification (Fixed / Decision / Excluded)

| Role | FacilityType | Count |
|---|---|---|
| **FIXED** | `NDC` | 1 |
| **DECISION** | `DEPO`, `RDC`, `DC Direct`, `DC Satellite` | 21 |
| **EXCLUDED** | `FC` (B2C only); `Instant Hub` (= `MFC` theo `Network_Map.md` chính thức của CEL: `DC Direct → MFC → B2C Customers`, chỉ phục vụ B2C và ko phát sinh chi phí do Paragon chi trả) | 5 |

**Tổng facility_model dùng cho MILP (FIXED + DECISION): 22**

> Instant Hub ban đầu bị nghi ngờ do 2 phát hiện thực nghiệm: (1) zero node phụ thuộc duy nhất vào nó trong coverage check B2B, (2) hoàn toàn vắng mặt như Origin hợp lệ trong bảng `CostLastMileB2B` (Unconstrained/Greenfield). Việc xác nhận Instant Hub = MFC (node B2C-only theo network map chính thức) giải thích trọn vẹn cả 2 phát hiện trên — nó chưa từng là facility B2B.


In [4]:
# ===== BƯỚC 1: Định nghĩa 3 nhóm facility theo vai trò trong model =====
# FIXED: bắt buộc luôn mở (NDC là node gốc nhận hàng từ factory, ko phải biến quyết định mở/đóng)
FIXED_TYPES = ["NDC"]

# DECISION: facility model sẽ quyết định mở hay đóng (biến nhị phân y_j trong MILP)
DECISION_TYPES = ["DEPO", "RDC", "DC Direct", "DC Satellite"]

# EXCLUDED: ngoài scope B2B distribution của thesis này (chỉ phục vụ B2C)
# - FC: chỉ phục vụ B2C
# - Instant Hub: = MFC theo Network_Map.md chính thức của CEL (DC Direct -> MFC -> B2C Customers),
#   cũng chỉ phục vụ B2C và ko phát sinh chi phí do Paragon chi trả (out of scope cho model B2B này)
EXCLUDED_TYPES = ["FC", "Instant Hub"]

# ===== BƯỚC 2: Gắn nhãn vai trò cho từng facility dựa theo FacilityType =====
def classify_facility(facility_type):
    if facility_type in FIXED_TYPES:
        return "FIXED"
    elif facility_type in DECISION_TYPES:
        return "DECISION"
    elif facility_type in EXCLUDED_TYPES:
        return "EXCLUDED"
    else:
        return "UNKNOWN"  # bắt lỗi nếu có FacilityType nào chưa được xếp loại

facility_master["FacilityRole"] = facility_master["FacilityType"].apply(classify_facility)

# ===== BƯỚC 3: Tách riêng từng nhóm + tạo bảng facility_model dùng cho MILP (loại EXCLUDED) =====
facility_fixed = facility_master[facility_master["FacilityRole"] == "FIXED"].copy()
facility_decision = facility_master[facility_master["FacilityRole"] == "DECISION"].copy()
facility_excluded = facility_master[facility_master["FacilityRole"] == "EXCLUDED"].copy()
facility_model = facility_master[facility_master["FacilityRole"] != "EXCLUDED"].copy()

# ===== SANITY CHECK =====
print("Phân bố FacilityRole:")
print(facility_master["FacilityRole"].value_counts())

print("\nCó FacilityType nào bị xếp UNKNOWN ko (nếu có nghĩa thiếu xử lý):")
print(facility_master[facility_master["FacilityRole"] == "UNKNOWN"][["LocationID", "FacilityType"]])

print("\nSố lượng theo từng nhóm:")
print("FIXED:", facility_fixed.shape[0])
print("DECISION:", facility_decision.shape[0])
print("EXCLUDED:", facility_excluded.shape[0])
print("Tổng dùng cho model (FIXED + DECISION):", facility_model.shape[0])

print("\nHead(10) facility_model:")
print(facility_model.head(10))

Phân bố FacilityRole:
FacilityRole
DECISION    21
EXCLUDED     5
FIXED        1
Name: count, dtype: int64

Có FacilityType nào bị xếp UNKNOWN ko (nếu có nghĩa thiếu xử lý):
Empty DataFrame
Columns: [LocationID, FacilityType]
Index: []

Số lượng theo từng nhóm:
FIXED: 1
DECISION: 21
EXCLUDED: 5
Tổng dùng cho model (FIXED + DECISION): 22

Head(10) facility_model:
   LocationID              FacilityName  FacilityType MainIsland  \
0         D09           DC Direct Bogor     DC Direct       Java   
1         D10               RDC Cirebon           RDC       Java   
2         D25            DC Direct Solo     DC Direct       Java   
3         D26        DC Direct Surabaya     DC Direct       Java   
4         D30                RDC Kediri           RDC       Java   
7     MWH_D04               RDC Bandung           RDC       Java   
8     MWH_D12                RDC Jember           RDC       Java   
9     MWH_D21   DC Satellite Purwokerto  DC Satellite       Java   
10    MWH_D23        DC 

In [7]:
import pandas as pd
import numpy as np
import os

# Đường dẫn gốc chứa data thô — XÁC NHẬN LẠI cho đúng
RAW_DATA_PATH = r"D:\KLTN\Data Dictionary\Modeling\Data"

# Đường dẫn folder xuất file đã clean
OUTPUT_PATH = r"D:\KLTN\Data Dictionary\Model - Data used"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# ===== BƯỚC 1: Load cost data Unconstrained/Greenfield (free-flow, không bị ép route lịch sử) =====
cost_lastmile = pd.read_parquet(os.path.join(RAW_DATA_PATH, "ENO_CostLastMileB2B_Unconstaint.parquet"))

# ===== BƯỚC 2: Lọc Origin chỉ giữ 22 facility trong facility_model (FC/Instant Hub bị loại) =====
valid_facility_ids = set(facility_model["LocationID"])
cost_lastmile_scoped = cost_lastmile[cost_lastmile["OriginID"].isin(valid_facility_ids)].copy()

# ===== BƯỚC 3: Đếm mỗi demand node có bao nhiêu facility phục vụ được (chiều node) =====
coverage_count = cost_lastmile_scoped.groupby("DestinationID")["OriginID"].nunique()

# ===== SANITY CHECK =====
print("Shape cost_lastmile (raw):", cost_lastmile.shape)
print(cost_lastmile.head(10))

print("\nShape cost_lastmile_scoped (Origin trong facility_model):", cost_lastmile_scoped.shape)
print(cost_lastmile_scoped.head(10))

# ===== Phân phối số-lượng-option mỗi node - đây là chỗ "bắt" vấn đề 95% hồi trước =====
print("\nPhân phối coverage_count (bao nhiêu node rơi vào từng mức số facility):")
print(coverage_count.value_counts().sort_index().head(10))

print("\nCoverage trung bình (facility/node):", coverage_count.mean())
print("% node chỉ có đúng 1 lựa chọn:", round((coverage_count == 1).mean() * 100, 2), "%")

# ===== Check node nào = 0 facility (infeasible nếu có) =====
uncovered = demand_agg[~demand_agg["ShipToGroupID"].isin(coverage_count.index)]
print(f"\nSố node KHÔNG có facility nào phục vụ: {len(uncovered)} / {len(demand_agg)}")
if len(uncovered) > 0:
    print(uncovered.head(10))

Shape cost_lastmile (raw): (22852, 3)
  OriginID                  DestinationID  CostPerPallet
0      D03  ID_BLN_Bali_Badung_Abiansemal   3.142993e+05
1      D03        ID_BLN_Bali_Badung_Kuta   2.735077e+05
2      D03      ID_BLN_Bali_Badung_Mengwi   2.545396e+05
3      D03      ID_BLN_Bali_Badung_Petang   8.572357e+05
4      D03      ID_BLN_Bali_Bangli_Bangli   8.523407e+05
5      D03   ID_BLN_Bali_Bangli_Kintamani   1.299213e+06
6      D03       ID_BLN_Bali_Bangli_Susut   9.761432e+05
7      D03     ID_BLN_Bali_Bangli_Tembuku   1.071596e+06
8      D03    ID_BLN_Bali_Buleleng_Banjar   1.491545e+06
9      D03  ID_BLN_Bali_Buleleng_Buleleng   1.577208e+06

Shape cost_lastmile_scoped (Origin trong facility_model): (10259, 3)
    OriginID                                      DestinationID  CostPerPallet
143      D09                                         105695-002   4.208002e+05
144      D09                                         128112-002   7.211374e+05
145      D09                

In [10]:
# ===== Check: facility đang cover mỗi node có phải là facility GẦN NHẤT không =====
# Dùng lại hàm haversine + cluster_centroids đã tính ở Section 4.3

# BƯỚC 1: hàm lấy toàn bộ 22 facility, sắp xếp theo khoảng cách thật tới 1 node
def get_nearest_facilities(shiptogroupid):
    coord = cluster_centroids[cluster_centroids["ShipToGroupID"] == shiptogroupid]
    dists = facility_model[["LocationID", "FacilityName"]].copy()
    dists["dist_km"] = haversine(
        facility_model["Latitude"], facility_model["Longitude"],
        coord["centroid_lat"].values[0], coord["centroid_lon"].values[0]
    )
    return dists.sort_values("dist_km")

# BƯỚC 2: với TOÀN BỘ 1,763 node, so sánh tập facility đang cover (cost data)
# với top-N facility gần nhất thật (N = số lượng đang cover)
match_results = []
for node_id, n_opt in coverage_count.items():
    nearest = get_nearest_facilities(node_id)
    covering = set(cost_lastmile_scoped[cost_lastmile_scoped["DestinationID"] == node_id]["OriginID"])
    top_n_nearest = set(nearest.head(n_opt)["LocationID"])
    match_results.append({
        "ShipToGroupID": node_id,
        "n_options": n_opt,
        "covering_set": covering,
        "top_n_nearest_set": top_n_nearest,
        "exact_match": covering == top_n_nearest,
        "overlap_pct": len(covering & top_n_nearest) / n_opt * 100
    })

match_df = pd.DataFrame(match_results)

# ===== SANITY CHECK =====
print("Shape match_df:", match_df.shape)
print(match_df.head(10))

print("\n% node mà facility cover TRÙNG KHỚP HOÀN TOÀN với top-N gần nhất:", round(match_df["exact_match"].mean() * 100, 2), "%")
print("Overlap trung bình (dù ko trùng 100%):", round(match_df["overlap_pct"].mean(), 2), "%")

# Xem thử vài node KHÔNG match, để hiểu vì sao lệch (có thể do rate rẻ hơn dù xa hơn 1 chút, ko hẳn là lỗi)
print("\nVí dụ node KHÔNG match hoàn toàn:")
print(match_df[~match_df["exact_match"]].head(10))

IndexError: index 0 is out of bounds for axis 0 with size 0

In [13]:
# ===== Check: facility đang cover mỗi node có phải là facility GẦN NHẤT không =====
def get_nearest_facilities(shiptogroupid):
    coord = cluster_centroids[cluster_centroids["ShipToGroupID"] == shiptogroupid]
    if coord.empty:
        return None  # node này không có centroid trong cluster_centroids
    dists = facility_model[["LocationID", "FacilityName"]].copy()
    dists["dist_km"] = haversine(
        facility_model["Latitude"], facility_model["Longitude"],
        coord["centroid_lat"].values[0], coord["centroid_lon"].values[0]
    )
    return dists.sort_values("dist_km")

# ===== So sánh, BỎ QUA node nào ko tìm được centroid, nhưng ĐẾM LẠI để biết mismatch bao nhiêu =====
match_results = []
missing_centroid_nodes = []

for node_id, n_opt in coverage_count.items():
    nearest = get_nearest_facilities(node_id)
    if nearest is None:
        missing_centroid_nodes.append(node_id)
        continue
    covering = set(cost_lastmile_scoped[cost_lastmile_scoped["DestinationID"] == node_id]["OriginID"])
    top_n_nearest = set(nearest.head(n_opt)["LocationID"])
    match_results.append({
        "ShipToGroupID": node_id,
        "n_options": n_opt,
        "exact_match": covering == top_n_nearest,
        "overlap_pct": len(covering & top_n_nearest) / n_opt * 100
    })

match_df = pd.DataFrame(match_results)

# ===== SANITY CHECK =====
print(f"Số node trong cost_lastmile_scoped KHÔNG có centroid trong cluster_centroids: {len(missing_centroid_nodes)} / {len(coverage_count)}")
print("Ví dụ vài node bị mismatch:", missing_centroid_nodes[:10])

print("\nShape match_df (chỉ tính trên node match được):", match_df.shape)
print(match_df.head(10))

print("\n% node mà facility cover TRÙNG KHỚP HOÀN TOÀN với top-N gần nhất:", round(match_df["exact_match"].mean() * 100, 2), "%")
print("Overlap trung bình:", round(match_df["overlap_pct"].mean(), 2), "%")

Số node trong cost_lastmile_scoped KHÔNG có centroid trong cluster_centroids: 113 / 1876
Ví dụ vài node bị mismatch: ['ID_BLN_Bali_Badung_Abiansemal', 'ID_BLN_Bali_Badung_Kuta', 'ID_BLN_Bali_Badung_Mengwi', 'ID_BLN_Bali_Badung_Petang', 'ID_BLN_Bali_Bangli_Bangli', 'ID_BLN_Bali_Bangli_Kintamani', 'ID_BLN_Bali_Bangli_Susut', 'ID_BLN_Bali_Bangli_Tembuku', 'ID_BLN_Bali_Buleleng_Banjar', 'ID_BLN_Bali_Buleleng_Buleleng']

Shape match_df (chỉ tính trên node match được): (1763, 4)
                         ShipToGroupID  n_options  exact_match  overlap_pct
0                           105695-002          7         True        100.0
1                           118438-598          8        False         87.5
2                           124624-000          7         True        100.0
3                           128112-002          7         True        100.0
4                           138789-000          7         True        100.0
5                           150794-001          7         True    

In [ ]:
# ===== Filter thêm: Destination cũng phải nằm trong scope Java (demand_agg) =====
valid_shipto_ids = set(demand_agg["ShipToGroupID"])
cost_lastmile_scoped = cost_lastmile_scoped[cost_lastmile_scoped["DestinationID"].isin(valid_shipto_ids)].copy()

print("Shape sau khi lọc cả Destination về Java-obnly:", cost_lastmile_scoped.shape)
print(cost_lastmile_scoped.head(10))

Shape sau khi lọc cả Destination về Java-only: (10080, 3)
    OriginID                                      DestinationID  CostPerPallet
143      D09                                         105695-002   4.208002e+05
144      D09                                         128112-002   7.211374e+05
145      D09                                         150794-001   1.890647e+05
146      D09                  ID_JV_Banten_Tangerang_Pagedangan   1.318201e+06
147      D09  ID_JV_Daerah Istimewa Yogyakarta_Bantul_Bangun...   1.767098e+07
148      D09     ID_JV_Daerah Istimewa Yogyakarta_Sleman_Godean   1.747601e+07
149      D09     ID_JV_Dki Jakarta_Kota Jakarta Pusat_Kemayoran   1.304743e+06
150      D09       ID_JV_Dki Jakarta_Kota Jakarta Pusat_Menteng   1.221371e+06
151      D09   ID_JV_Dki Jakarta_Kota Jakarta Selatan_Jagakarsa   6.998020e+05
152      D09  ID_JV_Dki Jakarta_Kota Jakarta Selatan_Pesangg...   1.269294e+06


In [14]:
# ===== Tính lại coverage_count trên bảng đã lọc sạch =====
coverage_count = cost_lastmile_scoped.groupby("DestinationID")["OriginID"].nunique()

print("Phân phối coverage_count:")
print(coverage_count.value_counts().sort_index())

print("\nCoverage trung bình (facility/node):", coverage_count.mean())
print("% node chỉ có đúng 1 lựa chọn:", round((coverage_count == 1).mean() * 100, 2), "%")

Phân phối coverage_count:
OriginID
1     37
2     49
3     75
4    138
5    328
6    566
7    443
8    112
9     15
Name: count, dtype: int64

Coverage trung bình (facility/node): 5.717526942711287
% node chỉ có đúng 1 lựa chọn: 2.1 %


In [15]:
# ===== So sánh lại, giờ bảng đã sạch nên missing_centroid_nodes phải về 0 =====
match_results = []
missing_centroid_nodes = []

for node_id, n_opt in coverage_count.items():
    nearest = get_nearest_facilities(node_id)
    if nearest is None:
        missing_centroid_nodes.append(node_id)
        continue
    covering = set(cost_lastmile_scoped[cost_lastmile_scoped["DestinationID"] == node_id]["OriginID"])
    top_n_nearest = set(nearest.head(n_opt)["LocationID"])
    match_results.append({
        "ShipToGroupID": node_id,
        "n_options": n_opt,
        "exact_match": covering == top_n_nearest,
        "overlap_pct": len(covering & top_n_nearest) / n_opt * 100
    })

match_df = pd.DataFrame(match_results)

print(f"Số node KHÔNG có centroid (kỳ vọng giờ = 0): {len(missing_centroid_nodes)} / {len(coverage_count)}")
print("\nShape match_df:", match_df.shape)
print(match_df.head(10))
print("\n% match hoàn toàn:", round(match_df["exact_match"].mean() * 100, 2), "%")
print("Overlap trung bình:", round(match_df["overlap_pct"].mean(), 2), "%")

Số node KHÔNG có centroid (kỳ vọng giờ = 0): 0 / 1763

Shape match_df: (1763, 4)
                         ShipToGroupID  n_options  exact_match  overlap_pct
0                           105695-002          7         True        100.0
1                           118438-598          8        False         87.5
2                           124624-000          7         True        100.0
3                           128112-002          7         True        100.0
4                           138789-000          7         True        100.0
5                           150794-001          7         True        100.0
6                           230001-001          1         True        100.0
7    ID_JV_Banten_Kota Cilegon_Cibeber          5        False         80.0
8    ID_JV_Banten_Kota Cilegon_Cilegon          5        False         80.0
9  ID_JV_Banten_Kota Cilegon_Citangkil          5        False         80.0

% match hoàn toàn: 62.96 %
Overlap trung bình: 92.93 %


## 4.5 Distance & Coverage Feasibility

$Dist_{jk}$ via Haversine (interim, pending OSRM road-distance file). Coverage constraint uses $D_{max}=150$ km (Java B2B SLA, Paragon's Assumption Doc §4.7).


In [18]:
# ==============================================================================
# 4.5 DISTANCE & COVERAGE FEASIBILITY
# ==============================================================================

# ===== BƯỚC 1: Load cost data Unconstrained/Greenfield (free-flow, không bị ép route lịch sử) =====
cost_lastmile = pd.read_parquet(os.path.join(RAW_DATA_PATH, "ENO_CostLastMileB2B_Unconstaint.parquet"))

# ===== BƯỚC 2: Lọc CẢ Origin (facility) lẫn Destination (demand node) về đúng scope =====
# Origin: chỉ giữ 22 facility trong facility_model (FC/Instant Hub bị loại)
# Destination: chỉ giữ node Java trong demand_agg (nếu không lọc, sẽ lẫn cả node Bali)
valid_facility_ids = set(facility_model["LocationID"])
valid_shipto_ids = set(demand_agg["ShipToGroupID"])

cost_lastmile_scoped = cost_lastmile[
    cost_lastmile["OriginID"].isin(valid_facility_ids) &
    cost_lastmile["DestinationID"].isin(valid_shipto_ids)
].copy()

# ===== BƯỚC 3: Đếm mỗi demand node có bao nhiêu facility phục vụ được =====
coverage_count = cost_lastmile_scoped.groupby("DestinationID")["OriginID"].nunique()

# ===== SANITY CHECK 1: Coverage cơ bản =====
print("Shape cost_lastmile (raw):", cost_lastmile.shape)
print(cost_lastmile.head(10))

print("\nShape cost_lastmile_scoped (đã lọc cả Origin + Destination):", cost_lastmile_scoped.shape)
print(cost_lastmile_scoped.head(10))

print("\nPhân phối coverage_count (bao nhiêu node rơi vào từng mức số facility):")
print(coverage_count.value_counts().sort_index())

print("\nCoverage trung bình (facility/node):", coverage_count.mean())
print("% node chỉ có đúng 1 lựa chọn:", round((coverage_count == 1).mean() * 100, 2), "%")

# ===== SANITY CHECK 2: Node nào = 0 facility (infeasible nếu có) =====
uncovered = demand_agg[~demand_agg["ShipToGroupID"].isin(coverage_count.index)]
print(f"\nSố node KHÔNG có facility nào phục vụ: {len(uncovered)} / {len(demand_agg)}")
if len(uncovered) > 0:
    print(uncovered.head(10))

# ===== SANITY CHECK 3: Facility đang cover mỗi node có phải facility GẦN NHẤT thật không =====
# (Dùng lại hàm haversine + cluster_centroids đã có ở Section 4.3)
def get_nearest_facilities(shiptogroupid):
    coord = cluster_centroids[cluster_centroids["ShipToGroupID"] == shiptogroupid]
    if coord.empty:
        return None
    dists = facility_model[["LocationID", "FacilityName"]].copy()
    dists["dist_km"] = haversine(
        facility_model["Latitude"], facility_model["Longitude"],
        coord["centroid_lat"].values[0], coord["centroid_lon"].values[0]
    )
    return dists.sort_values("dist_km")

match_results = []
missing_centroid_nodes = []

for node_id, n_opt in coverage_count.items():
    nearest = get_nearest_facilities(node_id)
    if nearest is None:
        missing_centroid_nodes.append(node_id)
        continue
    covering = set(cost_lastmile_scoped[cost_lastmile_scoped["DestinationID"] == node_id]["OriginID"])
    top_n_nearest = set(nearest.head(n_opt)["LocationID"])
    match_results.append({
        "ShipToGroupID": node_id,
        "n_options": n_opt,
        "exact_match": covering == top_n_nearest,
        "overlap_pct": len(covering & top_n_nearest) / n_opt * 100
    })

match_df = pd.DataFrame(match_results)

print(f"\nSố node KHÔNG có centroid (kỳ vọng = 0): {len(missing_centroid_nodes)} / {len(coverage_count)}")
print("\nShape match_df:", match_df.shape)
print(match_df.head(10))
print("\n% node match hoàn toàn với top-N facility gần nhất:", round(match_df["exact_match"].mean() * 100, 2), "%")
print("Overlap trung bình:", round(match_df["overlap_pct"].mean(), 2), "%")

Shape cost_lastmile (raw): (22852, 3)
  OriginID                  DestinationID  CostPerPallet
0      D03  ID_BLN_Bali_Badung_Abiansemal   3.142993e+05
1      D03        ID_BLN_Bali_Badung_Kuta   2.735077e+05
2      D03      ID_BLN_Bali_Badung_Mengwi   2.545396e+05
3      D03      ID_BLN_Bali_Badung_Petang   8.572357e+05
4      D03      ID_BLN_Bali_Bangli_Bangli   8.523407e+05
5      D03   ID_BLN_Bali_Bangli_Kintamani   1.299213e+06
6      D03       ID_BLN_Bali_Bangli_Susut   9.761432e+05
7      D03     ID_BLN_Bali_Bangli_Tembuku   1.071596e+06
8      D03    ID_BLN_Bali_Buleleng_Banjar   1.491545e+06
9      D03  ID_BLN_Bali_Buleleng_Buleleng   1.577208e+06

Shape cost_lastmile_scoped (đã lọc cả Origin + Destination): (10080, 3)
    OriginID                                      DestinationID  CostPerPallet
143      D09                                         105695-002   4.208002e+05
144      D09                                         128112-002   7.211374e+05
145      D09             

## 4.6 Cost Parameters

Load và filter 4 bảng cost. **Mỗi bảng lấy đúng scenario nguồn** (xem lý do chọn Unconstrained/Greenfield ở Section 4.5):

| File | Scenario nguồn | Lý do |
|---|---|---|
| `ENO_CostLastMileB2B_Unconstaint.parquet` | Unconstrained/Greenfield | Free-flow cost, ko bị force bởi `Share*`; bảng Baseline chỉ phản ánh route lịch sử |
| `3_GF_Unconstrained/ENO_CostTransfer.parquet` | Unconstrained/Greenfield | Tương tự — free-flow transfer cost giữa các facility |
| `0,1_Baseline2025/ENO_CostSupply.parquet` | Baseline | Scenario-invariant (toàn bộ `CostPerPallet` = 0, Factory→NDC) |
| `0,1_Baseline2025/ENO_DOS.parquet` | Baseline | Ko phải bảng O-D nên ko bị ảnh hưởng bởi Constrained/Unconstrained |

Cả 2 bảng Unconstrained đều được filter chặt về đúng 22 facility trong `facility_model` — **KHÔNG** lấy thêm các candidate site Greenfield mới (giữ nguyên scope facility hiện hữu của thesis).


In [20]:
# ==============================================================================
# 4.6 COST PARAMETERS (Supply / Transfer / LastMileB2B) & DOS
# ==============================================================================

# ===== LastMileB2B: tái sử dụng luôn kết quả đã làm sạch ở 4.5, không load lại =====
cost_lastmile_final = cost_lastmile_scoped.copy()

# ===== Transfer: load từ folder Unconstrained/Greenfield, lọc CẢ Origin lẫn Destination =====
# (rút kinh nghiệm từ lỗi Bali ở 4.5 — phải lọc cả 2 chiều, không chỉ 1 chiều)
cost_transfer = pd.read_parquet(os.path.join(RAW_DATA_PATH, "3_GF_Unconstrained", "ENO_CostTransfer.parquet"))
transfer_final = cost_transfer[
    cost_transfer["OriginID"].isin(valid_facility_ids) &
    cost_transfer["DestinationID"].isin(valid_facility_ids)
].copy()

# ===== Supply: Factory -> NDC, luôn = 0 (đồng vị trí) - lấy bản Baseline (scenario-invariant) =====
cost_supply = pd.read_parquet(os.path.join(RAW_DATA_PATH, "0,1_Baseline2025", "ENO_CostSupply.parquet"))
supply_baseline = cost_supply.copy()

# ===== DOS: chỉ có trong folder 3_GF_Unconstrained (KHÔNG có trong 0,1_Baseline2025) =====
dos = pd.read_parquet(os.path.join(RAW_DATA_PATH, "3_GF_Unconstrained", "ENO_DOS.parquet"))
dos_agg = dos.groupby("LocationID")["DOS"].median().reset_index()

# Ghép vào đủ 22 facility trong facility_model - facility nào thiếu DOS (chủ yếu DEPO, cross-dock
# không tồn kho nhiều) thì gán = 0
dos_agg = facility_model[["LocationID", "FacilityName", "FacilityType"]].merge(
    dos_agg, on="LocationID", how="left"
)
dos_agg["DOS"] = dos_agg.apply(
    lambda r: 0 if pd.isna(r["DOS"]) and r["FacilityType"] == "DEPO" else r["DOS"], axis=1
)

# ===== SANITY CHECK =====
print("cost_lastmile_final shape:", cost_lastmile_final.shape)
print(cost_lastmile_final.head(10))

print("\ntransfer_final shape:", transfer_final.shape)
print(transfer_final.head(10))

print("\nsupply_baseline shape:", supply_baseline.shape)
print(supply_baseline.head(10))

print("\ndos_agg shape:", dos_agg.shape)
print(dos_agg.head(10))

print("\nFacility còn thiếu DOS sau khi áp assumption DEPO=0 (cần xem lại nếu có):")
print(dos_agg[dos_agg["DOS"].isna()])

cost_lastmile_final shape: (10080, 3)
    OriginID                                      DestinationID  CostPerPallet
143      D09                                         105695-002   4.208002e+05
144      D09                                         128112-002   7.211374e+05
145      D09                                         150794-001   1.890647e+05
146      D09                  ID_JV_Banten_Tangerang_Pagedangan   1.318201e+06
147      D09  ID_JV_Daerah Istimewa Yogyakarta_Bantul_Bangun...   1.767098e+07
148      D09     ID_JV_Daerah Istimewa Yogyakarta_Sleman_Godean   1.747601e+07
149      D09     ID_JV_Dki Jakarta_Kota Jakarta Pusat_Kemayoran   1.304743e+06
150      D09       ID_JV_Dki Jakarta_Kota Jakarta Pusat_Menteng   1.221371e+06
151      D09   ID_JV_Dki Jakarta_Kota Jakarta Selatan_Jagakarsa   6.998020e+05
152      D09  ID_JV_Dki Jakarta_Kota Jakarta Selatan_Pesangg...   1.269294e+06

transfer_final shape: (174, 5)
   OriginID DestinationID  DistanceInKM  CostPerPallet  IsExi

---
# 5. Solution Method

## 5.1 Robust Optimization Reformulation

> ✍️ **Write here:** explain the Bertsimas & Sim (2004) budget-of-uncertainty approach — how $\bar d_k$, $\hat d_k$, and $\Gamma$ combine into the robust counterpart constraint used in Section 3.2. State the chosen $\Gamma$ value or the sensitivity range tested, and why.

## 5.2 AUGMECON2

> ✍️ **Write here:** explain the augmented ε-constraint method with bypass (Mavrotas & Florios, 2013) — why it's used instead of plain ε-constraint or weighted sum, and how the grid/step size for $f_2$, $f_3$ is chosen.

## 5.3 TOPSIS

> ✍️ **Write here:** explain how the Pareto set is ranked post-hoc under different manager priority scenarios (cost-priority / environment-priority / balanced), and the criteria weights used.


---
# 6. Implementation

## 6.1 Pyomo Model


In [ ]:
# ==============================================================================
# 6.1 PYOMO MODEL BUILD — Robust MOMILP (f1: Total Cost only, "bản #1")
# ==============================================================================
import pyomo.environ as pyo

model = pyo.ConcreteModel(name="RobustMOMILP_f1")

# ---- SETS ----
model.J = pyo.Set(initialize=facility_model["LocationID"].tolist())
model.J_fixed = pyo.Set(within=model.J, initialize=facility_fixed["LocationID"].tolist())
model.J_decision = pyo.Set(within=model.J, initialize=facility_decision["LocationID"].tolist())
model.K = pyo.Set(initialize=demand_agg["ShipToGroupID"].tolist())
model.F = pyo.Set(initialize=supply_baseline["OriginID"].unique().tolist())

model.SupplyArcs = pyo.Set(dimen=2, initialize=list(
    supply_baseline[["OriginID", "DestinationID"]].itertuples(index=False, name=None)))
model.TransferArcs = pyo.Set(dimen=2, initialize=list(
    transfer_final[["OriginID", "DestinationID"]].itertuples(index=False, name=None)))
model.DeliveryArcs = pyo.Set(dimen=2, initialize=list(
    cost_lastmile_final[["OriginID", "DestinationID"]].itertuples(index=False, name=None)))

# ---- PARAMETERS ----
model.FixedCost = pyo.Param(model.J, initialize=facility_model.set_index("LocationID")["FixedStorageCost_perMonth"].to_dict(), default=0)

# Capacity goc (ton kho tinh) - giu lai de tham khao, KHONG dung truc tiep trong constraint nua
model.Capacity = pyo.Param(model.J, initialize=facility_model.set_index("LocationID")["CapacityInPallet"].to_dict(), default=0)

# EffectiveCapacity = Capacity * 30 / DOS: quy doi ton kho tinh -> thong luong THANG (throughput)
# DOS (Days of Supply) da tinh o 4.6 (dos_agg). Facility Capacity=0 (DEPO/cross-dock) -> EffectiveCapacity=0 -> dung Big-M
_dos_dict = dos_agg.set_index("LocationID")["DOS"].to_dict()
_cap_dict = facility_model.set_index("LocationID")["CapacityInPallet"].to_dict()

def _compute_eff_capacity(loc_id):
    cap = _cap_dict.get(loc_id, 0)
    dos = _dos_dict.get(loc_id, 0)
    if cap <= 0 or dos <= 0:
        return 0  # cross-dock (DEPO) hoac thieu DOS -> se dung Big-M trong constraint
    return cap * 30 / dos

_eff_capacity_dict = {j: _compute_eff_capacity(j) for j in facility_model["LocationID"]}
model.EffectiveCapacity = pyo.Param(model.J, initialize=_eff_capacity_dict, default=0)

model.SupplyCost = pyo.Param(model.SupplyArcs,
    initialize={(r.OriginID, r.DestinationID): r.CostPerPallet for r in supply_baseline.itertuples()}, default=0)
model.TransferCost = pyo.Param(model.TransferArcs,
    initialize={(r.OriginID, r.DestinationID): r.CostPerPallet for r in transfer_final.itertuples()}, default=0)
model.DeliveryCost = pyo.Param(model.DeliveryArcs,
    initialize={(r.OriginID, r.DestinationID): r.CostPerPallet for r in cost_lastmile_final.itertuples()}, default=0)

model.d_bar = pyo.Param(model.K, initialize=demand_agg.set_index("ShipToGroupID")["d_bar"].to_dict())
model.d_hat = pyo.Param(model.K, initialize=demand_agg.set_index("ShipToGroupID")["d_hat"].to_dict())
model.Gamma = pyo.Param(initialize=0.5, mutable=True)

# ---- DECISION VARIABLES ----
model.Y = pyo.Var(model.J, within=pyo.Binary)
model.Supply = pyo.Var(model.SupplyArcs, within=pyo.NonNegativeReals)
model.Transfer = pyo.Var(model.TransferArcs, within=pyo.NonNegativeReals)
model.Delivery = pyo.Var(model.DeliveryArcs, within=pyo.NonNegativeReals)
model.z = pyo.Var(model.K, within=pyo.NonNegativeReals)
model.p = pyo.Var(model.K, within=pyo.NonNegativeReals)

for j in model.J_fixed:
    model.Y[j].fix(1)

# ---- OBJECTIVE: f1 - TOTAL COST ----
def total_cost_rule(m):
    fixed_cost = sum(m.FixedCost[j] * m.Y[j] for j in m.J_decision)
    supply_cost = sum(m.SupplyCost[i, j] * m.Supply[i, j] for (i, j) in m.SupplyArcs)
    transfer_cost = sum(m.TransferCost[j, l] * m.Transfer[j, l] for (j, l) in m.TransferArcs)
    delivery_cost = sum(m.DeliveryCost[j, k] * m.Delivery[j, k] for (j, k) in m.DeliveryArcs)
    return fixed_cost + supply_cost + transfer_cost + delivery_cost

model.f1_TotalCost = pyo.Objective(rule=total_cost_rule, sense=pyo.minimize)

# ---- CONSTRAINTS ----
# (1) Can bang dong tai moi facility
def flow_balance_rule(m, j):
    inflow = sum(m.Supply[i, jj] for (i, jj) in m.SupplyArcs if jj == j) \
           + sum(m.Transfer[l, jj] for (l, jj) in m.TransferArcs if jj == j)
    outflow = sum(m.Transfer[jj, l] for (jj, l) in m.TransferArcs if jj == j) \
            + sum(m.Delivery[jj, k] for (jj, k) in m.DeliveryArcs if jj == j)
    return inflow == outflow

model.FlowBalance = pyo.Constraint(model.J, rule=flow_balance_rule)

# (2) Suc chua + lien ket voi Y_j - dung EffectiveCapacity (DOS-adjusted throughput)
# thay vi Capacity tinh; DEPO/cross-dock (EffectiveCapacity=0) dung Big-M
BIG_M = demand_agg["d_bar"].sum() + demand_agg["d_hat"].sum()

def capacity_link_rule(m, j):
    outflow = sum(m.Transfer[jj, l] for (jj, l) in m.TransferArcs if jj == j) \
            + sum(m.Delivery[jj, k] for (jj, k) in m.DeliveryArcs if jj == j)
    eff_cap = pyo.value(m.EffectiveCapacity[j])
    if eff_cap > 0:
        return outflow <= m.EffectiveCapacity[j] * m.Y[j]
    else:
        return outflow <= BIG_M * m.Y[j]

model.CapacityLink = pyo.Constraint(model.J, rule=capacity_link_rule)

# (3) Robust demand satisfaction (Bertsimas & Sim counterpart)
def robust_demand_rule(m, k):
    served = sum(m.Delivery[j, kk] for (j, kk) in m.DeliveryArcs if kk == k)
    return served >= m.d_bar[k] + m.Gamma * m.z[k] + m.p[k]

model.RobustDemand = pyo.Constraint(model.K, rule=robust_demand_rule)

def robust_dual_rule(m, k):
    return m.z[k] + m.p[k] >= m.d_hat[k]

model.RobustDual = pyo.Constraint(model.K, rule=robust_dual_rule)

# ---- SANITY CHECK ----
n_vars = sum(1 for _ in model.component_data_objects(pyo.Var))
n_cons = sum(1 for _ in model.component_data_objects(pyo.Constraint))
print(f"So bien: {n_vars} | So rang buoc: {n_cons}")
print(f"BIG_M: {BIG_M:,.2f}")

So bien: 13806 | So rang buoc: 3570
BIG_M: 11,864.74


In [38]:
# ==============================================================================
# 6.1b TEST SOLVE — deterministic case (Gamma = 0)
# ==============================================================================
model.Gamma.set_value(0)
solver = pyo.SolverFactory("appsi_highs")
results = solver.solve(model, tee=False)

print(f"Solver status: {results.solver.status}")
print(f"Termination condition: {results.solver.termination_condition}")
if results.solver.termination_condition == pyo.TerminationCondition.optimal:
    print(f"Total Cost (f1) = {pyo.value(model.f1_TotalCost):,.2f}")
    n_open = sum(1 for j in model.J_decision if pyo.value(model.Y[j]) > 0.5)
    print(f"So facility DECISION duoc mo: {n_open}/{len(model.J_decision)}")

Solver status: ok
Termination condition: optimal
Total Cost (f1) = 11,317,795,608.62
So facility DECISION duoc mo: 17/21


In [39]:
# ==============================================================================
# 6.1m KIEM TRA NGHIEM: facility nao dong, breakdown chi phi
# ==============================================================================
# Facility DECISION bi dong (Y=0)
closed_facilities = [j for j in model.J_decision if pyo.value(model.Y[j]) < 0.5]
print("--- Facility DECISION bi dong (Y=0) ---")
print(facility_model[facility_model["LocationID"].isin(closed_facilities)]
      [["LocationID", "FacilityName", "FacilityType", "FixedStorageCost_perMonth", "CapacityInPallet"]])

# Breakdown chi phi theo tung thanh phan trong f1
fixed_cost_val = sum(pyo.value(model.FixedCost[j]) * pyo.value(model.Y[j]) for j in model.J_decision)
supply_cost_val = sum(pyo.value(model.SupplyCost[i, j]) * pyo.value(model.Supply[i, j]) for (i, j) in model.SupplyArcs)
transfer_cost_val = sum(pyo.value(model.TransferCost[j, l]) * pyo.value(model.Transfer[j, l]) for (j, l) in model.TransferArcs)
delivery_cost_val = sum(pyo.value(model.DeliveryCost[j, k]) * pyo.value(model.Delivery[j, k]) for (j, k) in model.DeliveryArcs)

print(f"\n--- Breakdown Total Cost ---")
print(f"Fixed cost (facility mo): {fixed_cost_val:,.2f}")
print(f"Supply cost (Factory->NDC): {supply_cost_val:,.2f}")
print(f"Transfer cost (facility->facility): {transfer_cost_val:,.2f}")
print(f"Delivery cost (facility->demand): {delivery_cost_val:,.2f}")
print(f"Tong (kiem tra khop f1): {fixed_cost_val+supply_cost_val+transfer_cost_val+delivery_cost_val:,.2f}")

--- Facility DECISION bi dong (Y=0) ---
   LocationID                 FacilityName FacilityType  \
0         D09              DC Direct Bogor    DC Direct   
4         D30                   RDC Kediri          RDC   
13    MWH_D36             DC Direct Banten    DC Direct   
24     DEPO08  Depo Bojonegoro - DC Kediri         DEPO   

    FixedStorageCost_perMonth  CapacityInPallet  
0                4.635301e+08             296.0  
4                7.861220e+08             502.0  
13               5.590549e+08             357.0  
24               2.933676e+07               0.0  

--- Breakdown Total Cost ---
Fixed cost (facility mo): 3,730,698,176.51
Supply cost (Factory->NDC): 0.00
Transfer cost (facility->facility): 2,766,796,097.41
Delivery cost (facility->demand): 4,820,301,334.71
Tong (kiem tra khop f1): 11,317,795,608.62


In [40]:
# ==============================================================================
# 6.1n SENSITIVITY: quet Gamma tu 0 den 1, xem network thay doi the nao
# ==============================================================================
gamma_values = [0.0, 0.25, 0.5, 0.75, 1.0]
sensitivity_results = []

for g in gamma_values:
    model.Gamma.set_value(g)
    # load_solutions=False de tranh crash neu co muc Gamma nao infeasible
    res = solver.solve(model, tee=False, load_solutions=False)
    term_cond = res.solver.termination_condition

    if term_cond == pyo.TerminationCondition.optimal:
        model.solutions.load_from(res)
        total_cost = pyo.value(model.f1_TotalCost)
        open_set = frozenset(j for j in model.J_decision if pyo.value(model.Y[j]) > 0.5)
        n_open = len(open_set)
    else:
        total_cost = None
        n_open = None
        open_set = frozenset()

    sensitivity_results.append({
        "Gamma": g,
        "Termination": str(term_cond),
        "TotalCost": total_cost,
        "N_Open": n_open,
        "OpenFacilities": open_set
    })

sensitivity_df = pd.DataFrame(sensitivity_results)

# Tinh % tang chi phi so voi baseline Gamma=0 (gia cua "su an toan")
baseline_cost = sensitivity_df.loc[sensitivity_df["Gamma"] == 0.0, "TotalCost"].values[0]
sensitivity_df["CostIncrease_vs_Gamma0_pct"] = (
    (sensitivity_df["TotalCost"] - baseline_cost) / baseline_cost * 100
)

print("--- Ket qua sensitivity theo Gamma ---")
print(sensitivity_df[["Gamma", "Termination", "TotalCost", "N_Open", "CostIncrease_vs_Gamma0_pct"]].head(10))

# Xem facility nao THAY DOI trang thai (mo them / dong bot) khi Gamma tang, so voi Gamma=0
baseline_open = sensitivity_df.loc[sensitivity_df["Gamma"] == 0.0, "OpenFacilities"].values[0]
print("\n--- Facility thay doi trang thai so voi Gamma=0 ---")
for row in sensitivity_results:
    if row["Gamma"] == 0.0:
        continue
    newly_opened = row["OpenFacilities"] - baseline_open
    newly_closed = baseline_open - row["OpenFacilities"]
    print(f"Gamma={row['Gamma']}: mo them={sorted(newly_opened)} | dong bot={sorted(newly_closed)}")

--- Ket qua sensitivity theo Gamma ---
   Gamma Termination     TotalCost  N_Open  CostIncrease_vs_Gamma0_pct
0   0.00     optimal  1.131780e+10      17                    0.000000
1   0.25     optimal  1.263252e+10      17                   11.616445
2   0.50     optimal  1.410635e+10      17                   24.638682
3   0.75     optimal  1.541691e+10      19                   36.218353
4   1.00     optimal  1.677004e+10      19                   48.174050

--- Facility thay doi trang thai so voi Gamma=0 ---
Gamma=0.25: mo them=[] | dong bot=[]
Gamma=0.5: mo them=[] | dong bot=[]
Gamma=0.75: mo them=['D30', 'DEPO08'] | dong bot=[]
Gamma=1.0: mo them=['D30', 'DEPO08'] | dong bot=[]


In [29]:
# ==============================================================================
# 6.1b TEST SOLVE — deterministic case (Gamma = 0) để kiểm tra model feasible
# ==============================================================================
model.Gamma.set_value(0)

solver = pyo.SolverFactory("appsi_highs")
results = solver.solve(model, tee=False)

print(f"Solver status: {results.solver.status}")
print(f"Termination condition: {results.solver.termination_condition}")

if results.solver.termination_condition == pyo.TerminationCondition.optimal:
    print(f"Total Cost (f1) = {pyo.value(model.f1_TotalCost):,.2f}")
    n_open = sum(1 for j in model.J_decision if pyo.value(model.Y[j]) > 0.5)
    print(f"So facility DECISION duoc mo: {n_open}/{len(model.J_decision)}")

RuntimeError: A feasible solution was not found, so no solution can be loaded. If using the appsi.solvers.Highs interface, you can set opt.config.load_solution=False. If using the environ.SolverFactory interface, you can set opt.solve(model, load_solutions = False). Then you can check results.termination_condition and results.best_feasible_objective before loading a solution.

## Diagnosis

In [30]:
# ==============================================================================
# 6.1i DIAGNOSE: kiem tra reachability tu NDC qua TransferArcs (BFS)
# ==============================================================================
# BƯỚC 1: solve an toan de lay termination_condition, khong crash
results = solver.solve(model, tee=False, load_solutions=False)
print(f"Termination condition: {results.solver.termination_condition}")

# BƯỚC 2: build graph tu TransferArcs, BFS tu NDC de tim tat ca facility "voi toi duoc"
from collections import defaultdict, deque

graph = defaultdict(list)
for (j, l) in transfer_final[["OriginID", "DestinationID"]].itertuples(index=False, name=None):
    graph[j].append(l)

ndc_id = facility_fixed["LocationID"].iloc[0]
visited = {ndc_id}
queue = deque([ndc_id])
while queue:
    node = queue.popleft()
    for nxt in graph[node]:
        if nxt not in visited:
            visited.add(nxt)
            queue.append(nxt)

print(f"\nSo facility 'voi toi' duoc tu NDC qua Transfer (ke ca NDC): {len(visited)} / {len(facility_model)}")

# BƯỚC 3: facility nao co DeliveryArc (phai giao hang) nhung KHONG nam trong 'visited'
# -> facility nay khong bao gio nhan duoc hang, bat ke Capacity/Big-M
facilities_with_delivery = set(cost_lastmile_final["OriginID"].unique())
unreachable_but_deliver = facilities_with_delivery - visited
print(f"So facility co the giao hang nhung KHONG the voi toi tu NDC: {len(unreachable_but_deliver)}")
print(facility_model[facility_model["LocationID"].isin(unreachable_but_deliver)]
      [["LocationID", "FacilityName", "FacilityType", "CapacityInPallet"]])

# BƯỚC 4: demand node nao CHI duoc phuc vu boi cac facility khong-voi-toi-duoc nay
if len(unreachable_but_deliver) > 0:
    stuck_nodes = cost_lastmile_final.groupby("DestinationID")["OriginID"].apply(
        lambda origins: set(origins).issubset(unreachable_but_deliver)
    )
    print(f"\nSo demand node chac chan infeasible (bi 'ket' do mat ket noi): {stuck_nodes.sum()}")
    print(stuck_nodes[stuck_nodes].head(10))

Termination condition: infeasible

So facility 'voi toi' duoc tu NDC qua Transfer (ke ca NDC): 22 / 22
So facility co the giao hang nhung KHONG the voi toi tu NDC: 0
Empty DataFrame
Columns: [LocationID, FacilityName, FacilityType, CapacityInPallet]
Index: []


In [31]:
# ==============================================================================
# 6.1j DIAGNOSE: relax toan bo capacity, mo het facility, chi giu Flow + RobustDemand
# ==============================================================================
# Tạo 1 model test riêng (khong dung chung voi model chinh, tranh anh huong)
test_model = pyo.ConcreteModel(name="RelaxTest")

test_model.J = pyo.Set(initialize=facility_model["LocationID"].tolist())
test_model.K = pyo.Set(initialize=demand_agg["ShipToGroupID"].tolist())
test_model.SupplyArcs = pyo.Set(dimen=2, initialize=list(supply_baseline[["OriginID","DestinationID"]].itertuples(index=False, name=None)))
test_model.TransferArcs = pyo.Set(dimen=2, initialize=list(transfer_final[["OriginID","DestinationID"]].itertuples(index=False, name=None)))
test_model.DeliveryArcs = pyo.Set(dimen=2, initialize=list(cost_lastmile_final[["OriginID","DestinationID"]].itertuples(index=False, name=None)))

test_model.d_bar = pyo.Param(test_model.K, initialize=demand_agg.set_index("ShipToGroupID")["d_bar"].to_dict())

# Bien: KHONG co Y (mo het mac dinh), chi co dong hang
test_model.Supply = pyo.Var(test_model.SupplyArcs, within=pyo.NonNegativeReals)
test_model.Transfer = pyo.Var(test_model.TransferArcs, within=pyo.NonNegativeReals)
test_model.Delivery = pyo.Var(test_model.DeliveryArcs, within=pyo.NonNegativeReals)

# Flow balance - giu nguyen logic nhu model chinh
def tb_rule(m, j):
    inflow = sum(m.Supply[i, jj] for (i, jj) in m.SupplyArcs if jj == j) \
           + sum(m.Transfer[l, jj] for (l, jj) in m.TransferArcs if jj == j)
    outflow = sum(m.Transfer[jj, l] for (jj, l) in m.TransferArcs if jj == j) \
            + sum(m.Delivery[jj, k] for (jj, k) in m.DeliveryArcs if jj == j)
    return inflow == outflow
test_model.FlowBalance = pyo.Constraint(test_model.J, rule=tb_rule)

# Demand satisfaction DON GIAN (khong robust, chi can >= d_bar) - de test thuan tuy capacity/topology
def td_rule(m, k):
    served = sum(m.Delivery[j, kk] for (j, kk) in m.DeliveryArcs if kk == k)
    return served >= m.d_bar[k]
test_model.DemandSat = pyo.Constraint(test_model.K, rule=td_rule)

# Objective gia (feasibility test, khong quan tam cost)
test_model.obj = pyo.Objective(expr=sum(test_model.Delivery[a] for a in test_model.DeliveryArcs), sense=pyo.minimize)

# Solve
test_solver = pyo.SolverFactory("appsi_highs")
test_results = test_solver.solve(test_model, tee=False, load_solutions=False)
print(f"Termination condition (relax toan bo): {test_results.solver.termination_condition}")

Termination condition (relax toan bo): optimal


In [32]:
# ==============================================================================
# 6.1k: them CapacityLink (voi fix Big-M) vao test_model, van mo het facility
# ==============================================================================
capacity_dict = facility_model.set_index("LocationID")["CapacityInPallet"].to_dict()
BIG_M = demand_agg["d_bar"].sum() + demand_agg["d_hat"].sum()

def tc_rule(m, j):
    outflow = sum(m.Transfer[jj, l] for (jj, l) in m.TransferArcs if jj == j) \
            + sum(m.Delivery[jj, k] for (jj, k) in m.DeliveryArcs if jj == j)
    cap = capacity_dict.get(j, 0)
    limit = cap if cap > 0 else BIG_M
    return outflow <= limit

test_model.CapacityLink = pyo.Constraint(test_model.J, rule=tc_rule)

test_results2 = test_solver.solve(test_model, tee=False, load_solutions=False)
print(f"Termination condition (co CapacityLink, van mo het facility): {test_results2.solver.termination_condition}")

Termination condition (co CapacityLink, van mo het facility): infeasible


In [33]:
# ==============================================================================
# DIAGNOSE: tim facility nao la bottleneck that (khi CapacityLink lam infeasible)
# ==============================================================================
# Thu "mo capacity" (thay bang Big-M) cho TUNG facility mot, giai lai
# facility nao lam model chuyen tu infeasible -> optimal chinh la nut that (bottleneck)
bottleneck_candidates = []

for j_test in [j for j in test_model.J if capacity_dict.get(j, 0) > 0]:
    def tc_rule_relaxed(m, j, relaxed_j=j_test):
        outflow = sum(m.Transfer[jj, l] for (jj, l) in m.TransferArcs if jj == j) \
                + sum(m.Delivery[jj, k] for (jj, k) in m.DeliveryArcs if jj == j)
        cap = capacity_dict.get(j, 0)
        limit = BIG_M if (j == relaxed_j or cap <= 0) else cap
        return outflow <= limit

    trial_model = test_model.clone()
    trial_model.del_component(trial_model.CapacityLink)
    trial_model.CapacityLink = pyo.Constraint(trial_model.J, rule=tc_rule_relaxed)
    trial_res = test_solver.solve(trial_model, tee=False, load_solutions=False)
    if trial_res.solver.termination_condition == pyo.TerminationCondition.optimal:
        bottleneck_candidates.append(j_test)

print(f"Facility ma neu no capacity se giai phong feasibility: {bottleneck_candidates}")
print(facility_model[facility_model["LocationID"].isin(bottleneck_candidates)]
      [["LocationID", "FacilityName", "FacilityType", "CapacityInPallet"]])

Facility ma neu no capacity se giai phong feasibility: []
Empty DataFrame
Columns: [LocationID, FacilityName, FacilityType, CapacityInPallet]
Index: []


In [34]:
# ==============================================================================
# DIAGNOSE: chuyen sang max-flow style - cho phep thieu hang (Unmet_k), minimize tong thieu
# de tim CHINH XAC node nao bi nghen do capacity that
# ==============================================================================
shortfall_model = pyo.ConcreteModel(name="ShortfallTest")

shortfall_model.J = pyo.Set(initialize=facility_model["LocationID"].tolist())
shortfall_model.K = pyo.Set(initialize=demand_agg["ShipToGroupID"].tolist())
shortfall_model.SupplyArcs = pyo.Set(dimen=2, initialize=list(supply_baseline[["OriginID","DestinationID"]].itertuples(index=False, name=None)))
shortfall_model.TransferArcs = pyo.Set(dimen=2, initialize=list(transfer_final[["OriginID","DestinationID"]].itertuples(index=False, name=None)))
shortfall_model.DeliveryArcs = pyo.Set(dimen=2, initialize=list(cost_lastmile_final[["OriginID","DestinationID"]].itertuples(index=False, name=None)))
shortfall_model.d_bar = pyo.Param(shortfall_model.K, initialize=demand_agg.set_index("ShipToGroupID")["d_bar"].to_dict())

shortfall_model.Supply = pyo.Var(shortfall_model.SupplyArcs, within=pyo.NonNegativeReals)
shortfall_model.Transfer = pyo.Var(shortfall_model.TransferArcs, within=pyo.NonNegativeReals)
shortfall_model.Delivery = pyo.Var(shortfall_model.DeliveryArcs, within=pyo.NonNegativeReals)
# Bien thieu hang: cho phep khong dap ung du d_bar, nhung se bi phat trong objective
shortfall_model.Unmet = pyo.Var(shortfall_model.K, within=pyo.NonNegativeReals)

def sf_flow_rule(m, j):
    inflow = sum(m.Supply[i, jj] for (i, jj) in m.SupplyArcs if jj == j) \
           + sum(m.Transfer[l, jj] for (l, jj) in m.TransferArcs if jj == j)
    outflow = sum(m.Transfer[jj, l] for (jj, l) in m.TransferArcs if jj == j) \
            + sum(m.Delivery[jj, k] for (jj, k) in m.DeliveryArcs if jj == j)
    return inflow == outflow
shortfall_model.FlowBalance = pyo.Constraint(shortfall_model.J, rule=sf_flow_rule)

def sf_cap_rule(m, j):
    outflow = sum(m.Transfer[jj, l] for (jj, l) in m.TransferArcs if jj == j) \
            + sum(m.Delivery[jj, k] for (jj, k) in m.DeliveryArcs if jj == j)
    cap = capacity_dict.get(j, 0)
    limit = cap if cap > 0 else BIG_M
    return outflow <= limit
shortfall_model.CapacityLink = pyo.Constraint(shortfall_model.J, rule=sf_cap_rule)

def sf_demand_rule(m, k):
    served = sum(m.Delivery[j, kk] for (j, kk) in m.DeliveryArcs if kk == k)
    return served + m.Unmet[k] >= m.d_bar[k]
shortfall_model.DemandSat = pyo.Constraint(shortfall_model.K, rule=sf_demand_rule)

# Objective: minimize tong luong thieu hang (chi de tim ra cho bi nghen, khong quan tam cost)
shortfall_model.obj = pyo.Objective(expr=sum(shortfall_model.Unmet[k] for k in shortfall_model.K), sense=pyo.minimize)

sf_results = test_solver.solve(shortfall_model, tee=False)
print(f"Termination: {sf_results.solver.termination_condition}")
print(f"Tong Unmet (luong khong the dap ung do capacity that): {pyo.value(shortfall_model.obj):,.2f}")

# Liet ke cac node bi thieu hang, sap xep giam dan
unmet_df = pd.DataFrame({
    "ShipToGroupID": list(shortfall_model.K),
    "Unmet": [pyo.value(shortfall_model.Unmet[k]) for k in shortfall_model.K]
})
unmet_df = unmet_df[unmet_df["Unmet"] > 1e-6].sort_values("Unmet", ascending=False)
print(f"\nSo node bi thieu hang: {len(unmet_df)}")
print(unmet_df.head(10))

Termination: optimal
Tong Unmet (luong khong the dap ung do capacity that): 655.40

So node bi thieu hang: 363
                                      ShipToGroupID      Unmet
1362              ID_JV_Jawa Timur_Jombang_Sumobito  37.557877
1676             ID_JV_Jawa Timur_Sidoarjo_Gedangan  34.584315
1127                ID_JV_Jawa Tengah_Sragen_Sragen  33.928636
172   ID_JV_Daerah Istimewa Yogyakarta_Sleman_Depok  33.428762
113                ID_JV_Banten_Tangerang_Tigaraksa  31.525434
1432         ID_JV_Jawa Timur_Kota Surabaya_Rungkut  28.856871
920             ID_JV_Jawa Tengah_Klaten_Karanganom  24.736414
964     ID_JV_Jawa Tengah_Kota Surakarta_Banjarsari  20.566515
1754        ID_JV_Jawa Timur_Tulungagung_Kedungwaru  19.076213
978              ID_JV_Jawa Tengah_Kudus_Kota Kudus  17.074761


In [35]:
# ==============================================================================
# DIAGNOSE: kiem tra gia thuyet - dung DOS de chuyen Capacity (ton kho) -> Throughput (thang)
# ==============================================================================
# Xem lai dos_agg da build o 4.6
print(dos_agg.sort_values("DOS").head(10))
print(f"\nThong ke DOS: min={dos_agg['DOS'].min()}, median={dos_agg['DOS'].median()}, max={dos_agg['DOS'].max()}")

# Tinh Effective Monthly Throughput = Capacity * 30 / DOS (chi ap dung cho facility co DOS > 0 va Capacity > 0)
dos_dict = dos_agg.set_index("LocationID")["DOS"].to_dict()

def effective_capacity(loc_id):
    cap = capacity_dict.get(loc_id, 0)
    dos = dos_dict.get(loc_id, 0)
    if cap <= 0:
        return 0  # DEPO/cross-dock: van dung Big-M nhu cu, khong lien quan DOS
    if dos <= 0:
        return cap  # fallback: neu khong co DOS (khong nen xay ra voi facility co Capacity>0), giu nguyen
    return cap * 30 / dos

eff_capacity_dict = {j: effective_capacity(j) for j in facility_model["LocationID"]}
print("\n--- So sanh Capacity goc vs Effective Throughput (DOS-adjusted) ---")
compare_df = facility_model[facility_model["CapacityInPallet"] > 0][["LocationID","FacilityName","FacilityType","CapacityInPallet"]].copy()
compare_df["DOS"] = compare_df["LocationID"].map(dos_dict)
compare_df["EffectiveThroughput"] = compare_df["LocationID"].map(eff_capacity_dict)
print(compare_df.sort_values("EffectiveThroughput"))

   LocationID                         FacilityName  FacilityType  DOS
12     DEPO04               Depo Bekasi - DC Bogor          DEPO  0.0
13     DEPO02             Depo Tegal - DC Semarang          DEPO  0.0
14     DEPO12              Depo Pati - DC Semarang          DEPO  0.0
15     DEPO18            Depo Yogyakarta - DC Solo          DEPO  0.0
19     DEPO08          Depo Bojonegoro - DC Kediri          DEPO  0.0
18     DEPO07              Depo Madiun - DC Kediri          DEPO  0.0
17     DEPO17            Depo Malang - DC Surabaya          DEPO  0.0
16     DEPO13  Depo Madura Pamekasan - DC Surabaya          DEPO  0.0
20     DEPO03             Depo Jakarta - DC Banten          DEPO  0.0
9     MWH_D27             DC Satellite Tasikmalaya  DC Satellite  7.0

Thong ke DOS: min=0.0, median=7.0, max=11.2

--- So sanh Capacity goc vs Effective Throughput (DOS-adjusted) ---
   LocationID              FacilityName  FacilityType  CapacityInPallet   DOS  \
12    MWH_D35     DC Satellite Suka

In [36]:
# ==============================================================================
# VERIFY: chay lai shortfall model voi Effective Throughput (DOS-adjusted) thay vi Capacity goc
# ==============================================================================
def sf_cap_rule_v2(m, j):
    outflow = sum(m.Transfer[jj, l] for (jj, l) in m.TransferArcs if jj == j) \
            + sum(m.Delivery[jj, k] for (jj, k) in m.DeliveryArcs if jj == j)
    eff_cap = eff_capacity_dict.get(j, 0)
    limit = eff_cap if eff_cap > 0 else BIG_M
    return outflow <= limit

shortfall_model.del_component(shortfall_model.CapacityLink)
shortfall_model.CapacityLink = pyo.Constraint(shortfall_model.J, rule=sf_cap_rule_v2)

sf_results_v2 = test_solver.solve(shortfall_model, tee=False)
print(f"Termination: {sf_results_v2.solver.termination_condition}")
print(f"Tong Unmet sau khi dieu chinh theo DOS: {pyo.value(shortfall_model.obj):,.2f}")

# Kiem tra con node nao thieu hang khong
unmet_df_v2 = pd.DataFrame({
    "ShipToGroupID": list(shortfall_model.K),
    "Unmet": [pyo.value(shortfall_model.Unmet[k]) for k in shortfall_model.K]
})
unmet_df_v2 = unmet_df_v2[unmet_df_v2["Unmet"] > 1e-6].sort_values("Unmet", ascending=False)
print(f"\nSo node con thieu hang: {len(unmet_df_v2)}")
print(unmet_df_v2.head(10))

Termination: optimal
Tong Unmet sau khi dieu chinh theo DOS: 0.00

So node con thieu hang: 0
Empty DataFrame
Columns: [ShipToGroupID, Unmet]
Index: []


## 6.2 AUGMECON2 Solve Loop


In [ ]:
# TODO: AUGMECON2 implementation


## 6.3 TOPSIS Ranking


In [ ]:
# TODO: TOPSIS ranking on Pareto set


---
# 7. Results & Discussion

> ✍️ **Write here** once the model runs: Pareto front summary, selected scenarios from TOPSIS, sensitivity to $\Gamma$, network configuration decisions (facility open/close), and comparison to the current (baseline) network.

*(placeholder — fill in after running Section 6)*
